# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yashcodes07/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1 **— ML Appendix, Feature Importance for Health Score (Random Forest: Average Position 43%, Impressions 32%, Scroll Depth 15%)
My methodology question: The paper's own "Understanding the Metrics" section discloses that Health Score is a composite literally computed as Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts). Three of the top four "predictive" features in this Random Forest — Position, Impressions, and implicitly CTR — are direct arithmetic components of the target itself. To the paper's credit, it discloses this plainly ("health score is partly constructed from these inputs... importance is descriptive rather than causal"), which is exactly the right move. My question is about the validation design specifically: an 80/20 holdout split is reported, but holding out rows doesn't address a target that's a deterministic function of some of its own inputs — no amount of held-out data can make "Position predicts a formula that includes Position" surprising. Would a cleaner test have been to exclude the constituent features entirely and check whether the remaining variables (content age, word count, search volume) carry any real signal for health score — a narrower but more honest question the current setup can't actually answer?

**Finding 2** — ML Appendix, Growth Prediction (Logistic Regression, 71% holdout accuracy; Content Age, Days Since Update, and Days Visible as top signals)
My methodology question: The paper's own metric glossary defines trend_direction as calculated from "30d-vs-prev-30d impression change" — i.e., the label is a comparison within a fixed, already-elapsed 30-day window, not a genuine future-outcome label. This is the same shape of proxy label I used in my own Week-1/Week-8 assignments, and it's exactly the kind of label my own audit called out as needing a future-window definition to support a real predictive claim. The Methodology section states an 80/20 split was used for this model but doesn't specify whether it's time-aware, grouped by brand, or a plain random split — and with 57 brands in the portfolio, a random split risks the same brand appearing in both train and test, letting the model partly learn brand-specific baselines rather than a generalizable growth pattern. Given days_with_impressions correlates with content_age_days at r=0.496 (per the paper's own correlation matrix), and the label is itself an impression-based comparison, is there a risk that "Days Visible" is partly measuring the same underlying visibility history the label is built from, rather than independently forecasting it? Since the paper doesn't disclose the split strategy for this specific model, I can't tell whether this concern is already ruled out or genuinely open.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Yashcodes07/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())


import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
df.columns.tolist()
print(df.columns.tolist())

Working dir: /content/flyrank-ml-internship
(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [2]:
paper_citations = {
    "finding_1": "ML Appendix — Feature Importance (p.27): Average Position 43%, "
                 "Impressions 32%, Scroll Depth 15%. Health Score formula (p.5): "
                 "Impressions(30) + Position(30) + CTR(20) + Scroll Depth(20).",
    "finding_2": "ML Appendix — Growth & Classification (p.29): Logistic Regression, "
                 "71% holdout accuracy. Trend Direction definition (p.5): "
                 "'Calculated from 30d-vs-prev-30d impression change.' "
                 "Correlation matrix (p.25): content_age_days x days_with_impressions r=0.496.",
}
for k, v in paper_citations.items():
    print(f"{k}:\n  {v}\n")


finding_1:
  ML Appendix — Feature Importance (p.27): Average Position 43%, Impressions 32%, Scroll Depth 15%. Health Score formula (p.5): Impressions(30) + Position(30) + CTR(20) + Scroll Depth(20).

finding_2:
  ML Appendix — Growth & Classification (p.29): Logistic Regression, 71% holdout accuracy. Trend Direction definition (p.5): 'Calculated from 30d-vs-prev-30d impression change.' Correlation matrix (p.25): content_age_days x days_with_impressions r=0.496.



## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
feature_cols = ["ctr", "content_age_days", "impressions_90d"]  # match your real Week-5 list
target_col = "trend_direction"

def precision_at_50(clf, X_test, test_df):
    probs = clf.predict_proba(X_test)
    down_idx = list(clf.classes_).index("down")
    scored = test_df.copy()
    scored["down_prob"] = probs[:, down_idx]
    top50 = scored.sort_values("down_prob", ascending=False).head(50)
    return (top50[target_col] == "down").mean()

# BEFORE: naive random split
train_r, test_r = train_test_split(df, test_size=0.2, random_state=42, stratify=df[target_col])
clf_r = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight="balanced")
clf_r.fit(train_r[feature_cols], train_r[target_col])
p50_random = precision_at_50(clf_r, test_r[feature_cols], test_r)

# AFTER: grouped-by-client (or time-aware fallback)
if "client_id" in df.columns:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
    train_g, test_g = df.iloc[train_idx], df.iloc[test_idx]
else:
    date_col = next((c for c in df.columns if "date" in c.lower()), None)
    df_sorted = df.sort_values(date_col)
    cutoff = int(len(df_sorted) * 0.8)
    train_g, test_g = df_sorted.iloc[:cutoff], df_sorted.iloc[cutoff:]

clf_g = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight="balanced")
clf_g.fit(train_g[feature_cols], train_g[target_col])
p50_grouped = precision_at_50(clf_g, test_g[feature_cols], test_g)

print(f"BEFORE (naive random split): {p50_random:.3f}")
print(f"AFTER  (grouped/time-aware): {p50_grouped:.3f}")
print(f"Gap: {p50_random - p50_grouped:+.3f}")


BEFORE (naive random split): 0.780
AFTER  (grouped/time-aware): 0.460
Gap: +0.320


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
audit_notes = {
    "ctr": "Computed from clicks/impressions within the SAME window used to derive "
           "trend_direction. Same structural risk as the paper's Health Score finding: "
           "if the label is impression-derived and ctr is impression-derived, they share "
           "input, not independent measurement.",
    "content_age_days": "Static attribute, not derived from performance data — clean.",
    "impressions_90d": "Same window-overlap risk as ctr. This is the feature responsible "
                        "for the near-zero-traffic collision pattern found in ML-08 Section 4 "
                        "(down/flat/new pages sharing identical impressions_90d=3).",
}
for feat, note in audit_notes.items():
    print(f"{feat}:\n  {note}\n")

print("Mean impressions_90d by trend_direction:")
print(df.groupby("trend_direction")["impressions_90d"].mean().round(1))


ctr:
  Computed from clicks/impressions within the SAME window used to derive trend_direction. Same structural risk as the paper's Health Score finding: if the label is impression-derived and ctr is impression-derived, they share input, not independent measurement.

content_age_days:
  Static attribute, not derived from performance data — clean.

impressions_90d:
  Same window-overlap risk as ctr. This is the feature responsible for the near-zero-traffic collision pattern found in ML-08 Section 4 (down/flat/new pages sharing identical impressions_90d=3).

Mean impressions_90d by trend_direction:
trend_direction
down      4919.1
flat        20.7
new        164.1
stable    9213.6
up        4716.1
Name: impressions_90d, dtype: float64


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Boldest original claim**: "A learned model got about 74% right (0.740) on the same task — roughly 3x more true positives in the same review budget."
Rewritten in safe language:
"On a naive random split, the model showed markedly higher Precision@50 than the hand-written baseline (0.740 vs. 0.240). Under a grouped/time-aware split (Section 2), this gap narrowed to [insert your actual measured gap] — indicating part of the original lift reflected split leakage, not generalizable signal. The defensible claim: the model's ranked output is directionally more useful than the hand-written rule for prioritizing a review queue, and it measurably outperforms the rule under matched evaluation conditions — the specific '3x' figure should not be repeated without stating which split produced it. This mirrors the same caution the FlyRank paper applies to its own Health Score feature-importance finding: high performance is expected, and mildly interesting, only once you've confirmed it isn't circular or leaked."

In [5]:
print("Original numbers: baseline 0.240, model 0.740 (random split)")
print(f"Honest-split numbers: see Section 2 output above")
print("Claim rewritten to disclose which split each number came from — same standard I'm asking the paper to meet.")


Original numbers: baseline 0.240, model 0.740 (random split)
Honest-split numbers: see Section 2 output above
Claim rewritten to disclose which split each number came from — same standard I'm asking the paper to meet.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.